# Pruebas del Dispatch contra la DB

Este notebook prueba **el código real de `dispatch.py`** sobre los datos de MongoDB, paso por paso:

| Paso | En `dispatch.py` | Qué se revisa aquí |
|---|---|---|
| 0. Preparación de la DB | migración `003_digests` | que existan `digests`, su validador y el índice `por_llegada` |
| 1. Leer | `leer_noticias()` | qué noticias toma de `noticias` y por qué por `fetched_at` |
| 2. Escoger | `seleccionar()` | el texto exacto que recibe Claude y cuánto mide |
| 3. Escribir | `escribir()` | qué notas completas recibe Claude y el límite de 1,600 caracteres |
| 4. Guardar | `guardar_digest()` | que el digest que se guardaría pase el validador de `digests` |
| 5. Digests guardados | colección `digests` | integridad de los que ya existan |

**Claude está simulado**: no se usa la API ni hace falta la llave. La simulación escoge las 5 de mayor `score` y arma
un mensaje simple. Con eso se prueba todo el flujo de datos, pero no la calidad de lo que escribiría Claude.
**Este notebook no escribe nada en la DB.**

Corre todo con *Run All*. Puedes cambiar `HORAS` y `EXCLUIR_ARXIV` en la primera celda.

In [ ]:
import importlib.util
import json
import os
import re
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path
from pprint import pprint
from types import SimpleNamespace

import matplotlib.pyplot as plt
import pandas as pd

HORAS = 24              # la misma ventana que usa dispatch.py por defecto (--horas)
EXCLUIR_ARXIV = False   # dispatch.py hoy NO excluye arXiv; probar_con_claude_code.py sí

# Funciona igual en Docker (db/ en /app, el repo en /repo) que abierto directo desde db/testing/.
DB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "hedwix_db").is_dir())   # carpeta db/
RAIZ = next(p for p in [DB.parent, Path("/repo")] if (p / "dispatch" / "dispatch.py").is_file())  # raíz del repo
sys.path[:0] = [str(DB), str(RAIZ / "dispatch"), str(RAIZ / "funnel")]
if "MONGODB_URI" not in os.environ:
    # Fuera de Docker: se toma la URI del .env de la raíz y se conecta por 127.0.0.1 en lugar del host "mongo"
    from dotenv import dotenv_values
    os.environ["MONGODB_URI"] = dotenv_values(RAIZ / ".env")["MONGODB_URI"].replace("@mongo:", "@127.0.0.1:")

import dispatch
import funnel
from hedwix_db import obtener_db

db = obtener_db()
pd.set_option("display.max_colwidth", 90)
resultados, avisos = [], []

def chequeo(nombre, problemas, detalle=None):
    """Registra un chequeo. `problemas` es cuántos casos fallan (0 = bien)."""
    resultados.append({"chequeo": nombre, "problemas": problemas, "ok": problemas == 0})
    print(("✅" if problemas == 0 else "❌"), nombre, "" if problemas == 0 else f"→ {problemas} con problemas")
    if problemas and detalle is not None:
        display(detalle)

def aviso(texto):
    """Algo que conviene saber pero que no es un error de los datos."""
    avisos.append(texto)
    print("⚠️ ", texto)

def tokens_aprox(texto):
    return len(texto) // 4   # aproximación (~4 caracteres por token); el conteo exacto necesita la API

print(f"DB: {db.name} | dispatch.py: {RAIZ / 'dispatch' / 'dispatch.py'} | modelo configurado: {dispatch.MODELO}")

## 0. Preparación de la DB

`dispatch.py` necesita la migración `003_digests`, que crea la colección `digests` con su validador y el índice
`por_llegada` sobre `noticias.fetched_at`. Si el dispatch corre **antes** de la migración, Mongo crea `digests` sola al
guardar, pero **sin validador**. Ese es el caso que hay que evitar.

In [ ]:
spec = importlib.util.spec_from_file_location("m003", DB / "migraciones" / "003_digests.py")
m003 = importlib.util.module_from_spec(spec); spec.loader.exec_module(m003)

aplicadas = set(db["_migraciones"].distinct("_id"))
info = db.command("listCollections", filter={"name": "digests"})["cursor"]["firstBatch"]
existe = bool(info)
con_validador = existe and "validator" in info[0]["options"]
con_indice = "por_llegada" in db.noticias.index_information()

display(pd.DataFrame([
    {"qué": "migración 003_digests aplicada", "estado": "003_digests" in aplicadas},
    {"qué": "colección digests existe", "estado": existe},
    {"qué": "digests tiene validador", "estado": con_validador},
    {"qué": "índice noticias.por_llegada", "estado": con_indice},
]))

if "003_digests" not in aplicadas:
    aviso("La migración 003_digests no está aplicada en esta DB. Corre `docker compose exec app python migrar.py` "
          "(desde db/) antes de usar dispatch.py. Mientras tanto, este notebook valida contra el esquema del archivo "
          "de la migración.")
chequeo("digests no existe sin validador", 1 if existe and not con_validador else 0)
if "003_digests" in aplicadas:
    chequeo("Índice por_llegada creado por la migración", 0 if con_indice else 1)

# El esquema contra el que se valida: el de la DB si ya está, si no el de la migración
ESQUEMA_DIGESTS = info[0]["options"]["validator"]["$jsonSchema"] if con_validador else m003.ESQUEMA
iguales = ESQUEMA_DIGESTS == m003.ESQUEMA
chequeo("El validador de la DB es igual al de la migración 003", 0 if iguales else 1)

## 1. Leer: `leer_noticias()`

El dispatch toma las noticias que **llegaron** (`fetched_at`) en las últimas `HORAS`, no las que se **publicaron**
(`published`) en ese tiempo. Anthropic, Meta y HF Daily Papers sólo dan el día de publicación, así que una nota de
ayer guardada hoy se perdería si se leyera por `published`.

In [ ]:
desde = datetime.now(timezone.utc) - timedelta(hours=HORAS)
leidas = dispatch.leer_noticias(funnel.ARCHIVO_DE_SALIDA, desde)   # el mismo llamado que hace correr_dispatch()
origen = "MongoDB" if funnel.coleccion_de_noticias() is not None else "JSONL (¡no está usando la DB!)"
print(f"Ventana: desde {desde:%Y-%m-%d %H:%M} UTC ({HORAS} h) | origen: {origen} | leídas: {len(leidas)}")

chequeo("El dispatch está leyendo de MongoDB", 0 if origen == "MongoDB" else 1)
directo = db.noticias.count_documents({"fetched_at": {"$gte": desde}})
chequeo(f"leer_noticias() trae lo mismo que la consulta directa a Mongo ({directo})", abs(len(leidas) - directo))

CAMPOS_QUE_USA = ["id", "source", "title", "url", "summary", "content", "score"]
faltan = [n.get("id") for n in leidas if any(c not in n for c in CAMPOS_QUE_USA)]
chequeo(f"Cada noticia trae los campos que usa el dispatch {CAMPOS_QUE_USA}", len(faltan))

if not leidas:
    aviso(f"No hay noticias que hayan llegado en las últimas {HORAS} h: corre el funnel o sube HORAS. "
          "Los pasos 2 a 4 se saltan.")

In [ ]:
if leidas:
    df = pd.DataFrame(leidas)
    por_fuente = df.groupby("source").agg(noticias=("id", "size"),
                                          con_score=("score", lambda s: s.notna().sum()),
                                          sin_texto=("content", lambda s: (s == "").sum()))
    por_fuente = por_fuente.sort_values("noticias", ascending=False)
    arxiv = df["source"].str.startswith("arXiv").sum()
    print(f"{len(df)} noticias de {df['source'].nunique()} fuentes; arXiv: {arxiv} ({arxiv / len(df):.0%})")
    por_fuente["noticias"].sort_values().plot.barh(figsize=(8, 6), title="Lo que lee el dispatch, por fuente")
    plt.tight_layout(); plt.show()
    display(por_fuente)

In [ ]:
# ¿Qué se perdería si se leyera por published? (la razón del cambio a fetched_at)
por_published = db.noticias.count_documents({"published": {"$gte": desde}})
perdidas = list(db.noticias.aggregate([
    {"$match": {"fetched_at": {"$gte": desde}, "published": {"$lt": desde}}},
    {"$group": {"_id": "$source", "noticias": {"$sum": 1}}}, {"$sort": {"noticias": -1}},
]))
print(f"Por fetched_at entran {directo}; por published entrarían {por_published}.")
print(f"Llegaron en la ventana pero se publicaron antes (se perderían por published): {sum(p['noticias'] for p in perdidas)}")
display(pd.DataFrame(perdidas).rename(columns={"_id": "fuente"}) if perdidas else "ninguna")

# La consulta debe usar el índice por_llegada
etapa, etapas = db.noticias.find({"fetched_at": {"$gte": desde}}).explain()["queryPlanner"]["winningPlan"], []
while etapa:
    etapas.append(etapa["stage"] + (f"({etapa['indexName']})" if "indexName" in etapa else ""))
    etapa = etapa.get("inputStage")
plan = " <- ".join(etapas)
if con_indice:
    chequeo(f"La lectura usa el índice por_llegada: {plan}", 0 if "por_llegada" in plan else 1)
else:
    aviso(f"Sin el índice por_llegada la lectura recorre toda la colección: {plan}")

## 2. Escoger: `seleccionar()`

Aquí se ve el texto **exacto** que `dispatch.py` le manda a Claude para escoger, armado por su propio código. Para eso
se reemplaza sólo la llamada a la API (`preguntar_a_claude`) por una simulación que guarda la petición.

In [ ]:
capturas = []

def claude_simulado(cliente, instrucciones, mensajes, **opciones):
    """Reemplaza la llamada a la API: guarda lo que se le mandaría a Claude y responde algo plausible."""
    capturas.append({"instrucciones": instrucciones, "mensajes": mensajes, "opciones": opciones})
    if "format" in opciones.get("output_config", {}):            # paso 2: responde con el esquema de selección
        lista = mensajes[0]["content"]
        puntajes = {int(i): float(p or 0) for i, p in re.findall(r"^\[(\d+)\][^\n]*?(?:\| puntaje ([\d.]+))?$", lista, re.M)}
        mejores = sorted(puntajes, key=lambda i: -puntajes[i])[:dispatch.NOTICIAS_POR_DIGEST]
        texto = json.dumps({"seleccion": [{"indice": i, "por_que": "(simulado) de las de mayor puntaje"} for i in mejores]})
    else:                                                          # paso 3: un mensaje sencillo con títulos y links
        notas = mensajes[0]["content"]
        titulos = re.findall(r"^Título: (.*)$", notas, re.M)
        links = re.findall(r"^Link: (.*)$", notas, re.M)
        texto = "*Hedwix* (mensaje simulado)\n\n" + "\n\n".join(
            f"{n}. *{t[:60]}*\n{l}" for n, (t, l) in enumerate(zip(titulos, links), start=1))
    return SimpleNamespace(content=[SimpleNamespace(type="text", text=texto)])

dispatch.preguntar_a_claude = claude_simulado

def tamano_de_seleccion(noticias):
    """Corre seleccionar() y regresa (seleccionadas, texto que recibió Claude)."""
    seleccionadas = dispatch.seleccionar(None, noticias)
    return seleccionadas, capturas[-1]["mensajes"][0]["content"]

if leidas:
    sin_arxiv = [n for n in leidas if not n["source"].startswith("arXiv")]
    _, texto_todas = tamano_de_seleccion(leidas)
    _, texto_sin_arxiv = tamano_de_seleccion(sin_arxiv) if sin_arxiv else (None, "")
    display(pd.DataFrame([
        {"caso": "todas (lo que hace dispatch.py hoy)", "noticias": len(leidas),
         "caracteres": len(texto_todas), "tokens aprox.": tokens_aprox(texto_todas)},
        {"caso": "sin arXiv (como probar_con_claude_code.py)", "noticias": len(sin_arxiv),
         "caracteres": len(texto_sin_arxiv), "tokens aprox.": tokens_aprox(texto_sin_arxiv)},
    ]))

In [ ]:
if leidas:
    candidatas = [n for n in leidas if not (EXCLUIR_ARXIV and n["source"].startswith("arXiv"))]
    seleccionadas, texto_para_escoger = tamano_de_seleccion(candidatas)
    print(f"Instrucciones (system):\n{capturas[-1]['instrucciones'][:600]}…\n")
    print(f"Lista de noticias (primeros 1,500 de {len(texto_para_escoger):,} caracteres):\n")
    print(texto_para_escoger[:1500] + "\n…")

In [ ]:
if leidas:
    ids = [n["id"] for n in seleccionadas]
    chequeo(f"seleccionar() regresa {dispatch.NOTICIAS_POR_DIGEST} noticias distintas",
            abs(len(set(ids)) - dispatch.NOTICIAS_POR_DIGEST))
    chequeo("Las escogidas existen en la colección noticias",
            len(set(ids)) - db.noticias.count_documents({"_id": {"$in": ids}}))
    chequeo("Cada escogida trae su por_que", sum(1 for n in seleccionadas if not n.get("por_que")))
    print("\nEscogidas (por la simulación, no por Claude):")
    display(pd.DataFrame(seleccionadas)[["source", "title", "score", "url"]])

## 3. Escribir: `escribir()`

Claude sólo recibe el texto completo (`content`) de las escogidas. Si una nota no tiene texto, se usa su `summary`.
El mensaje final no puede pasar de 1,600 caracteres, que es el límite de Twilio para WhatsApp.

In [ ]:
if leidas:
    fecha = datetime.now(dispatch.ZONA_HORARIA).date().isoformat()
    mensaje = dispatch.escribir(None, seleccionadas, fecha)
    notas = capturas[-1]["mensajes"][0]["content"]
    print(f"Fecha del digest (México): {fecha}")
    print(f"Lo que recibe Claude para escribir: {len(notas):,} caracteres (~{tokens_aprox(notas):,} tokens)\n")
    display(pd.DataFrame([{"source": n["source"], "title": n["title"][:60], "largo content": len(n["content"]),
                           "usa": "content" if n["content"] else "summary"} for n in seleccionadas]))
    chequeo(f"El mensaje mide como máximo {dispatch.LIMITE_DEL_MENSAJE} caracteres ({len(mensaje)})",
            0 if len(mensaje) <= dispatch.LIMITE_DEL_MENSAJE else 1)
    # Revisa todas las candidatas, no sólo las que escogió la simulación: Claude podría escoger cualquiera
    sin_texto = [n for n in candidatas if not (n["content"] or n["summary"])]
    if sin_texto:
        por_fuente_sin_texto = ", ".join(f"{f}: {n}" for f, n in pd.Series([n["source"] for n in sin_texto]).value_counts().items())
        aviso(f"{len(sin_texto)} candidatas no tienen content ni summary ({por_fuente_sin_texto}). Si Claude escoge "
              "una, escribe sólo con el título. HF trending nunca trae texto; en Hacker News pasa cuando el sitio "
              "bloquea la descarga.")
        display(pd.DataFrame(sin_texto)[["source", "title", "score", "url"]])
    print("\nMensaje (simulado):\n")
    print(mensaje)

## 4. Guardar: `guardar_digest()`, sin escribir en la DB

Se corre el `guardar_digest()` real, pero con la colección `digests` reemplazada por una que sólo guarda el documento.
Ese documento se valida contra el esquema de `digests` con `$documents`, que revisa sin insertar nada.

In [ ]:
def valida(documento):
    """True si el documento cumple el $jsonSchema de digests (se revisa sin insertarlo)."""
    fuera = list(db.aggregate([{"$documents": [documento]}, {"$match": {"$nor": [{"$jsonSchema": ESQUEMA_DIGESTS}]}}]))
    return not fuera

if leidas:
    class DigestsDePrueba:
        def __init__(self):
            self.documentos = []
        def replace_one(self, filtro, documento, upsert=False):
            self.documentos.append({**filtro, **documento})

    falsa = DigestsDePrueba()
    original = funnel.coleccion_de_noticias
    funnel.coleccion_de_noticias = lambda: SimpleNamespace(database={"digests": falsa})
    try:
        donde = dispatch.guardar_digest(fecha, mensaje, seleccionadas)
    finally:
        funnel.coleccion_de_noticias = original

    digest = falsa.documentos[0]
    print(f"guardar_digest() lo mandaría a: {donde}\n")
    pprint({**digest, "mensaje": digest["mensaje"][:80] + " […]"}, sort_dicts=False, width=110)
    print()
    chequeo("El digest que se guardaría cumple el validador de digests", 0 if valida(digest) else 1)
    chequeo("El _id del digest es la fecha AAAA-MM-DD", 0 if re.fullmatch(r"\d{4}-\d{2}-\d{2}", digest["_id"]) else 1)

    # Prueba en negativo: el validador sí rechaza lo que no debe entrar
    chequeo("El validador rechaza un mensaje de más de 1,600 caracteres",
            0 if not valida({**digest, "mensaje": "x" * 1601}) else 1)
    chequeo("El validador rechaza una noticia con id que no es sha1",
            0 if not valida({**digest, "noticias": [{**digest["noticias"][0], "id": "no-es-sha1"}]}) else 1)

## 5. Digests ya guardados

Revisa los digests que hayan escrito corridas reales de `dispatch.py`. Hoy no hay ninguno, porque necesitan la llave
de la API.

In [ ]:
digests = list(db.digests.find().sort("_id", -1)) if existe else []
if not digests:
    aviso("Todavía no hay digests guardados: se crean al correr dispatch.py con ANTHROPIC_API_KEY.")
else:
    # .get() porque aquí también se revisan digests mal formados (por ejemplo, creados antes de la migración)
    display(pd.DataFrame([{"fecha": d["_id"], "generado_at": d.get("generado_at"), "modelo": d.get("modelo"),
                           "caracteres": len(d.get("mensaje", "")), "noticias": len(d.get("noticias", []))}
                          for d in digests]))
    invalidos = [d["_id"] for d in digests if not valida(d)]
    chequeo("Todos los digests cumplen el validador", len(invalidos))

    # Integridad: cada noticia de un digest debe existir en noticias
    huerfanas = list(db.digests.aggregate([
        {"$unwind": "$noticias"},
        {"$lookup": {"from": "noticias", "localField": "noticias.id", "foreignField": "_id", "as": "n"}},
        {"$match": {"n": {"$size": 0}}},
        {"$project": {"fecha": "$_id", "id": "$noticias.id", "title": "$noticias.title", "_id": 0}},
    ]))
    chequeo("Cada noticia de un digest existe en noticias", len(huerfanas), pd.DataFrame(huerfanas))

    # Informativo: noticias que se repiten entre digests
    repetidas = list(db.digests.aggregate([
        {"$unwind": "$noticias"},
        {"$group": {"_id": "$noticias.id", "title": {"$first": "$noticias.title"}, "fechas": {"$push": "$_id"}}},
        {"$match": {"fechas.1": {"$exists": True}}},
    ]))
    print(f"ℹ️  {len(repetidas)} noticias aparecen en más de un digest")
    if repetidas:
        display(pd.DataFrame(repetidas))
    print(f"\nÚltimo digest ({digests[0]['_id']}):\n\n{digests[0].get('mensaje', '(sin mensaje)')}")

## Resumen

In [ ]:
resumen = pd.DataFrame(resultados)
print(f"{resumen['ok'].sum()} de {len(resumen)} chequeos OK | {len(avisos)} avisos")
for texto in avisos:
    print("⚠️ ", texto)
resumen.style.map(lambda ok: "background-color: #d1fae5" if ok else "background-color: #fee2e2", subset=["ok"])

In [ ]:
# Para correrlo desde la terminal: si algún chequeo falla, el notebook termina con error (los avisos no fallan)
assert resumen["ok"].all(), "Hay chequeos con problemas: " + ", ".join(resumen.loc[~resumen["ok"], "chequeo"])